In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------
# Example Dataset (replace with your real dataset)
# -------------------------------
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# Advanced Forecast View Model Pipeline
# -------------------------------
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LogisticRegression(max_iter=500))
])

# Hyperparameter tuning (important for accuracy boost)
param_grid = {
    "model__C": [0.1, 1, 10, 50],
    "model__solver": ["lbfgs", "liblinear"]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

# -------------------------------
# Evaluation
# -------------------------------
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Best Parameters:", grid.best_params_)
print("Test Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'model__C': 10, 'model__solver': 'lbfgs'}
Test Accuracy: 0.9666666666666667

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.90      0.95        10
           2       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [2]:
import unittest
import numpy as np

class TestForecastViewModel(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        global best_model
        cls.model = best_model

        cls.sample_input = np.array([X_test[0]])

    # 1
    def test_prediction_shape(self):
        pred = self.model.predict(self.sample_input)
        self.assertEqual(len(pred), 1)

    # 2
    def test_output_valid_class(self):
        pred = self.model.predict(self.sample_input)[0]
        self.assertIn(pred, [0, 1, 2])

    # 3
    def test_accuracy_threshold(self):
        self.assertGreaterEqual(accuracy, 0.90)

    # 4
    def test_consistency(self):
        p1 = self.model.predict(self.sample_input)
        p2 = self.model.predict(self.sample_input)
        self.assertEqual(p1[0], p2[0])

    # 5
    def test_batch_prediction(self):
        preds = self.model.predict(X_test[:10])
        self.assertEqual(len(preds), 10)

    # 6
    def test_no_nan_output(self):
        preds = self.model.predict(X_test)
        self.assertFalse(np.isnan(preds).any())

    # 7
    def test_robustness_noise(self):
        noisy = X_test[0] + np.random.normal(0, 0.1, size=X_test[0].shape)
        pred = self.model.predict([noisy])
        self.assertIn(pred[0], [0, 1, 2])

    # 8
    def test_boundary_values(self):
        extreme = np.array([X_test.max(axis=0)])
        pred = self.model.predict(extreme)
        self.assertIn(pred[0], [0, 1, 2])

    # 9
    def test_training_output_exists(self):
        self.assertIsNotNone(best_model)

    # 10
    def test_prediction_determinism(self):
        preds1 = self.model.predict(X_test[:5])
        preds2 = self.model.predict(X_test[:5])
        np.testing.assert_array_equal(preds1, preds2)

    # 11 (extra)
    def test_single_feature_variation(self):
        modified = X_test.copy()
        modified[0][0] += 0.5
        pred = self.model.predict(modified[:1])
        self.assertIn(pred[0], [0, 1, 2])


if __name__ == "__main__":
    unittest.main()

E
ERROR: /root/ (unittest.loader._FailedTest./root/)
----------------------------------------------------------------------
AttributeError: module '__main__' has no attribute '/root/'

----------------------------------------------------------------------
Ran 1 test in 0.007s

FAILED (errors=1)


SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
import unittest
import numpy as np

# Assumes these already exist from your training cell:
# best_model, X_test, y_test, accuracy

class TestForecastViewModel(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.model = best_model
        cls.X_test = X_test
        cls.y_test = y_test

    # 1. Check prediction shape
    def test_prediction_shape(self):
        pred = self.model.predict(self.X_test[:1])
        self.assertEqual(pred.shape[0], 1)

    # 2. Check valid class labels
    def test_valid_classes(self):
        pred = self.model.predict(self.X_test[:10])
        for p in pred:
            self.assertIn(p, np.unique(self.y_test))

    # 3. Accuracy threshold (≥ 90%)
    def test_accuracy_threshold(self):
        self.assertGreaterEqual(accuracy, 0.90)

    # 4. Consistency test
    def test_consistency(self):
        p1 = self.model.predict(self.X_test[:1])
        p2 = self.model.predict(self.X_test[:1])
        self.assertEqual(p1[0], p2[0])

    # 5. Batch prediction test
    def test_batch_prediction(self):
        pred = self.model.predict(self.X_test[:20])
        self.assertEqual(len(pred), 20)

    # 6. No NaN predictions
    def test_no_nan(self):
        pred = self.model.predict(self.X_test)
        self.assertFalse(np.isnan(pred).any())

    # 7. Robustness to noise
    def test_noise_robustness(self):
        noisy = self.X_test[0] + np.random.normal(0, 0.1, size=self.X_test[0].shape)
        pred = self.model.predict([noisy])
        self.assertIn(pred[0], np.unique(self.y_test))

    # 8. Boundary test (max values)
    def test_boundary_max_values(self):
        extreme = np.max(self.X_test, axis=0).reshape(1, -1)
        pred = self.model.predict(extreme)
        self.assertIn(pred[0], np.unique(self.y_test))

    # 9. Model existence test
    def test_model_exists(self):
        self.assertIsNotNone(self.model)

    # 10. Deterministic output test
    def test_deterministic(self):
        p1 = self.model.predict(self.X_test[:5])
        p2 = self.model.predict(self.X_test[:5])
        np.testing.assert_array_equal(p1, p2)

    # 11. Slight feature variation test
    def test_feature_variation(self):
        modified = self.X_test[:1].copy()
        modified[0][0] += 0.3
        pred = self.model.predict(modified)
        self.assertIn(pred[0], np.unique(self.y_test))


# -------------------------------
# SAFE RUN (IMPORTANT FOR NOTEBOOKS)
# -------------------------------
suite = unittest.TestLoader().loadTestsFromTestCase(TestForecastViewModel)
unittest.TextTestRunner(verbosity=2).run(suite)

test_accuracy_threshold (__main__.TestForecastViewModel.test_accuracy_threshold) ... ok
test_batch_prediction (__main__.TestForecastViewModel.test_batch_prediction) ... ok
test_boundary_max_values (__main__.TestForecastViewModel.test_boundary_max_values) ... ok
test_consistency (__main__.TestForecastViewModel.test_consistency) ... ok
test_deterministic (__main__.TestForecastViewModel.test_deterministic) ... ok
test_feature_variation (__main__.TestForecastViewModel.test_feature_variation) ... ok
test_model_exists (__main__.TestForecastViewModel.test_model_exists) ... ok
test_no_nan (__main__.TestForecastViewModel.test_no_nan) ... ok
test_noise_robustness (__main__.TestForecastViewModel.test_noise_robustness) ... ok
test_prediction_shape (__main__.TestForecastViewModel.test_prediction_shape) ... ok
test_valid_classes (__main__.TestForecastViewModel.test_valid_classes) ... ok

----------------------------------------------------------------------
Ran 11 tests in 0.085s

OK


<unittest.runner.TextTestResult run=11 errors=0 failures=0>